# Model B: EfficientNetB0 with geometric augmentation

Model A reached good clean-test performance, but its training curves showed a growing gap between training and validation performance. In this experiment, I test whether ordinary geometric augmentation reduces that overfitting.

I use horizontal flips, small rotations, translations, and zoom. I intentionally exclude brightness, contrast, gamma correction, and noise because illumination changes will be studied separately. Everything else—the data split, seed, backbone, classification head, callbacks, and clean test set—remains the same as Model A.

In [ ]:
from pathlib import Path
import json
import shutil
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from IPython.display import display
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0

SEED = 42
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

print(f"Python: {sys.version.split()[0]}")
print(f"TensorFlow: {tf.__version__}")
print(f"GPU available: {bool(tf.config.list_physical_devices('GPU'))}")

## Experiment settings

The augmentation ranges are deliberately moderate. Large rotations or translations could remove facial regions and change the task rather than simply add useful variation.

In [ ]:
CLASS_NAMES = ['angry', 'happy', 'sad']
DRIVE_DATASET_DIR = Path('/content/drive/MyDrive/Emotions Dataset')
LOCAL_DATASET_DIR = Path('/content/emotions_dataset')
PROJECT_OUTPUT_DIR = Path('/content/drive/MyDrive/CNN-Robustness-Low-Light-Analysis/outputs')
OUTPUT_DIR = PROJECT_OUTPUT_DIR / 'model_b'
MODEL_A_RESULTS_PATH = PROJECT_OUTPUT_DIR / 'model_a' / 'results_summary.json'

CONFIG = {
    'image_size': (224, 224),
    'batch_size': 32,
    'validation_split': 0.20,
    'epochs': 30,
    'learning_rate': 1e-3,
    'dropout_rate': 0.30,
    'dense_units': 256,
    'rotation_degrees': 10,
    'translation_fraction': 0.10,
    'zoom_fraction': 0.10,
}

CONFIG

## Prepare the dataset

As in Model A, I copy the files from Drive to the Colab runtime for faster reading. The original `train` folder is split into training and validation subsets, and the original `test` folder remains untouched until the final evaluation.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

required_directories = [
    DRIVE_DATASET_DIR / split / class_name
    for split in ['train', 'test']
    for class_name in CLASS_NAMES
]
missing_directories = [path for path in required_directories if not path.is_dir()]

if missing_directories:
    missing_text = '\n'.join(str(path) for path in missing_directories)
    raise FileNotFoundError(f"The following dataset folders were not found:\n{missing_text}")

if not LOCAL_DATASET_DIR.exists():
    print('Copying the dataset from Drive to the Colab runtime...')
    shutil.copytree(DRIVE_DATASET_DIR, LOCAL_DATASET_DIR)
    print('Copy complete.')
else:
    print('Using the dataset already copied to the Colab runtime.')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_DIR = LOCAL_DATASET_DIR / 'train'
TEST_DIR = LOCAL_DATASET_DIR / 'test'

if not MODEL_A_RESULTS_PATH.is_file():
    raise FileNotFoundError(
        f"Model A results were not found at {MODEL_A_RESULTS_PATH}. "
        "Run the Model A notebook first."
    )

with open(MODEL_A_RESULTS_PATH) as file:
    model_a_results = json.load(file)

print(f"Model A clean-test accuracy: {model_a_results['test_accuracy']:.4f}")
print(f"Model A clean-test macro-F1: {model_a_results['test_macro_f1']:.4f}")

In [ ]:
common_dataset_options = {
    'image_size': CONFIG['image_size'],
    'batch_size': CONFIG['batch_size'],
    'label_mode': 'categorical',
    'class_names': CLASS_NAMES,
}

train_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=CONFIG['validation_split'],
    subset='training',
    seed=SEED,
    shuffle=True,
    **common_dataset_options,
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=CONFIG['validation_split'],
    subset='validation',
    seed=SEED,
    shuffle=True,
    **common_dataset_options,
)

test_dataset = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    shuffle=False,
    **common_dataset_options,
)

test_file_paths = list(test_dataset.file_paths)
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.prefetch(AUTOTUNE)
validation_dataset = validation_dataset.prefetch(AUTOTUNE)
test_dataset = test_dataset.prefetch(AUTOTUNE)

## Define and inspect the geometric augmentation

Keras augmentation layers are active only while the model is training. Validation and test images pass through unchanged. A horizontal flip is reasonable for facial expressions, whereas a vertical flip would create unrealistic faces and is therefore excluded.

In [ ]:
geometric_augmentation = tf.keras.Sequential(
    [
        layers.RandomFlip('horizontal', seed=SEED),
        layers.RandomRotation(
            factor=CONFIG['rotation_degrees'] / 360,
            fill_mode='reflect',
            seed=SEED + 1,
        ),
        layers.RandomTranslation(
            height_factor=CONFIG['translation_fraction'],
            width_factor=CONFIG['translation_fraction'],
            fill_mode='reflect',
            seed=SEED + 2,
        ),
        layers.RandomZoom(
            height_factor=(-CONFIG['zoom_fraction'], CONFIG['zoom_fraction']),
            width_factor=(-CONFIG['zoom_fraction'], CONFIG['zoom_fraction']),
            fill_mode='reflect',
            seed=SEED + 3,
        ),
    ],
    name='geometric_augmentation',
)

In [ ]:
sample_images, sample_labels = next(iter(train_dataset))
augmented_images = geometric_augmentation(sample_images[:6], training=True)

fig, axes = plt.subplots(2, 6, figsize=(15, 5))
for index in range(6):
    label_index = int(tf.argmax(sample_labels[index]))

    axes[0, index].imshow(sample_images[index].numpy().astype('uint8'))
    axes[0, index].set_title(CLASS_NAMES[label_index])
    axes[0, index].axis('off')

    clipped_image = tf.clip_by_value(augmented_images[index], 0, 255)
    axes[1, index].imshow(clipped_image.numpy().astype('uint8'))
    axes[1, index].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=12)
axes[1, 0].set_ylabel('Augmented', fontsize=12)
fig.suptitle('Geometric augmentation check')
fig.tight_layout()
plt.show()

## Define Model B

The only architectural difference from Model A is the augmentation block placed before EfficientNetB0. The pretrained backbone remains frozen, and EfficientNet performs its own input rescaling.

In [ ]:
tf.keras.utils.set_random_seed(SEED)

base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=CONFIG['image_size'] + (3,),
)
base_model.trainable = False

inputs = tf.keras.Input(shape=CONFIG['image_size'] + (3,), name='image')
augmented_inputs = geometric_augmentation(inputs)
features = base_model(augmented_inputs, training=False)
features = layers.GlobalAveragePooling2D(name='global_average_pooling')(features)
features = layers.Dropout(CONFIG['dropout_rate'], name='dropout')(features)
features = layers.Dense(CONFIG['dense_units'], activation='relu', name='classifier_dense')(features)
outputs = layers.Dense(len(CLASS_NAMES), activation='softmax', name='emotion')(features)

model = tf.keras.Model(inputs, outputs, name='model_b_geometric_augmentation')
model.summary()

trainable_parameters = sum(np.prod(variable.shape) for variable in model.trainable_weights)
print(f"Trainable parameters: {trainable_parameters:,}")

## Train with the same selection protocol as Model A

The best checkpoint is selected using validation loss. The untouched test set is still not used during training.

In [ ]:
checkpoint_path = OUTPUT_DIR / 'model_b_geometric_augmentation.keras'
training_log_path = OUTPUT_DIR / 'training_log.csv'

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=CONFIG['learning_rate']),
    loss='categorical_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.TopKCategoricalAccuracy(k=2, name='top_2_accuracy'),
    ],
)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(checkpoint_path),
        monitor='val_loss',
        mode='min',
        save_best_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        mode='min',
        patience=8,
        restore_best_weights=True,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        mode='min',
        factor=0.5,
        patience=4,
        min_lr=1e-7,
        verbose=1,
    ),
    tf.keras.callbacks.CSVLogger(str(training_log_path)),
]

In [ ]:
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=CONFIG['epochs'],
    callbacks=callbacks,
    verbose=2,
)

## Evaluate the saved checkpoint on clean test images

In [ ]:
best_model = tf.keras.models.load_model(str(checkpoint_path))
test_metrics = best_model.evaluate(test_dataset, return_dict=True, verbose=1)

print('\nClean test metrics')
for metric_name, metric_value in test_metrics.items():
    print(f"{metric_name}: {metric_value:.4f}")

In [ ]:
probabilities = best_model.predict(test_dataset, verbose=1)
predicted_labels = np.argmax(probabilities, axis=1)
true_labels = np.concatenate([
    np.argmax(batch_labels.numpy(), axis=1)
    for _, batch_labels in test_dataset
])

report = classification_report(
    true_labels,
    predicted_labels,
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0,
    output_dict=True,
)

print(classification_report(
    true_labels,
    predicted_labels,
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0,
))

predictions = pd.DataFrame({
    'file': test_file_paths,
    'true_label': [CLASS_NAMES[index] for index in true_labels],
    'predicted_label': [CLASS_NAMES[index] for index in predicted_labels],
})
for class_index, class_name in enumerate(CLASS_NAMES):
    predictions[f'prob_{class_name}'] = probabilities[:, class_index]

predictions.to_csv(OUTPUT_DIR / 'test_predictions.csv', index=False)

## Visualize the training behavior and class-level errors

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
epochs_ran = range(1, len(history.history['loss']) + 1)

axes[0].plot(epochs_ran, history.history['accuracy'], label='Training')
axes[0].plot(epochs_ran, history.history['val_accuracy'], label='Validation')
axes[0].set(title='Accuracy', xlabel='Epoch', ylabel='Accuracy')
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].plot(epochs_ran, history.history['loss'], label='Training')
axes[1].plot(epochs_ran, history.history['val_loss'], label='Validation')
axes[1].set(title='Cross-entropy loss', xlabel='Epoch', ylabel='Loss')
axes[1].legend()
axes[1].grid(alpha=0.25)

fig.suptitle('Model B training history')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'training_curves.png', dpi=200, bbox_inches='tight')
plt.show()

matrix = confusion_matrix(true_labels, predicted_labels)
plt.figure(figsize=(6, 5))
sns.heatmap(
    matrix,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
)
plt.title('Model B: clean test confusion matrix')
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=200, bbox_inches='tight')
plt.show()

## Compare Model B with Model A

This is a clean-test comparison only. Low-light robustness will be evaluated later using fixed corrupted versions of the same test images.

In [ ]:
model_b_accuracy = float(test_metrics['accuracy'])
model_b_macro_f1 = float(report['macro avg']['f1-score'])
accuracy_change = model_b_accuracy - float(model_a_results['test_accuracy'])
macro_f1_change = model_b_macro_f1 - float(model_a_results['test_macro_f1'])

comparison = pd.DataFrame({
    'Model': ['Model A: no augmentation', 'Model B: geometric augmentation'],
    'Accuracy': [model_a_results['test_accuracy'], model_b_accuracy],
    'Macro-F1': [model_a_results['test_macro_f1'], model_b_macro_f1],
})
display(comparison.style.format({'Accuracy': '{:.4f}', 'Macro-F1': '{:.4f}'}))

print(f"Accuracy change: {accuracy_change:+.4f}")
print(f"Macro-F1 change: {macro_f1_change:+.4f}")

comparison_plot = comparison.set_index('Model').plot.bar(
    figsize=(8, 5),
    ylim=(0, 1),
    rot=0,
    color=['#4C72B0', '#55A868'],
)
comparison_plot.set_ylabel('Score')
comparison_plot.set_title('Clean-test comparison')
comparison_plot.grid(axis='y', alpha=0.25)
comparison_plot.figure.tight_layout()
comparison_plot.figure.savefig(
    OUTPUT_DIR / 'comparison_with_model_a.png',
    dpi=200,
    bbox_inches='tight',
)
plt.show()

In [ ]:
history_to_save = {
    name: [float(value) for value in values]
    for name, values in history.history.items()
}

with open(OUTPUT_DIR / 'training_history.json', 'w') as file:
    json.dump(history_to_save, file, indent=2)

best_epoch = int(np.argmin(history.history['val_loss']) + 1)
results_summary = {
    'experiment': 'Model B - geometric augmentation',
    'seed': SEED,
    'tensorflow_version': tf.__version__,
    'image_size': list(CONFIG['image_size']),
    'batch_size': CONFIG['batch_size'],
    'validation_split': CONFIG['validation_split'],
    'augmentation': {
        'horizontal_flip': True,
        'rotation_degrees': CONFIG['rotation_degrees'],
        'translation_fraction': CONFIG['translation_fraction'],
        'zoom_fraction': CONFIG['zoom_fraction'],
        'photometric_transforms': False,
    },
    'epochs_completed': len(history.history['loss']),
    'best_epoch_by_validation_loss': best_epoch,
    'best_validation_loss': float(min(history.history['val_loss'])),
    'best_validation_accuracy': float(max(history.history['val_accuracy'])),
    'test_loss': float(test_metrics['loss']),
    'test_accuracy': model_b_accuracy,
    'test_top_2_accuracy': float(test_metrics['top_2_accuracy']),
    'test_macro_f1': model_b_macro_f1,
    'test_weighted_f1': float(report['weighted avg']['f1-score']),
    'model_a_test_accuracy': float(model_a_results['test_accuracy']),
    'model_a_test_macro_f1': float(model_a_results['test_macro_f1']),
    'accuracy_change_from_model_a': accuracy_change,
    'macro_f1_change_from_model_a': macro_f1_change,
    'classification_report': report,
}

with open(OUTPUT_DIR / 'results_summary.json', 'w') as file:
    json.dump(results_summary, file, indent=2)

print(json.dumps({
    'best_epoch': results_summary['best_epoch_by_validation_loss'],
    'test_accuracy': results_summary['test_accuracy'],
    'test_macro_f1': results_summary['test_macro_f1'],
    'accuracy_change_from_model_a': results_summary['accuracy_change_from_model_a'],
    'macro_f1_change_from_model_a': results_summary['macro_f1_change_from_model_a'],
}, indent=2))
print(f"\nSaved all Model B outputs to: {OUTPUT_DIR}")

## What to keep from this run

Keep the complete `outputs/model_b` folder. The most important files are `model_b_geometric_augmentation.keras`, `results_summary.json`, `training_history.json`, `training_curves.png`, `confusion_matrix.png`, and `comparison_with_model_a.png`.